### Dataset sintético tipo OBD

Este notebook construye el dataset sintético que se usa en canibalizacion_final.ipynb (y que se puede usar para correr los demás estimadores). Con los datos reales de OBD hay tres cosas que no se pueden hacer, que son conocer el $\beta$ verdadero, calcular el IPW de vitrina (porque random no sortea las vitrinas de forma uniforme y no conocemos la probabilidad de cada vitrina) y tener muestras tan grandes como queramos. Por eso se arma un dataset que copia la estructura de OBD (campaña *men*), pero donde todo eso es conocido.

**Ítems y clics.** Se usan los mismos 34 ítems con sus 7 categorías (item_feature_1). Hay 20 tipos de usuario, cada uno con 5 características $x$, y cada ítem tiene un vector $v_a$ de 5 componentes. La probabilidad de clic sin canibalización es
$$\theta_k(x, a) = \text{CTR}_a \cdot \text{efecto}_k \cdot e^{\sigma \langle x, v_a\rangle/\sqrt{5} - \sigma^2/2},$$
donde $\text{CTR}_a$ es el CTR del ítem en random y $\text{efecto}_k$ es el efecto de la posición, ambos sacados de OBD. El último factor hace que a cada tipo de usuario le gusten más unos ítems que otros, con $\sigma = 0{,}25$, y vale 1 en promedio. Con canibalización, los clics siguen el modelo lineal
$$P(\text{clic en } k) = \theta_k(x, a_k) - \beta c_k, \qquad \beta = s \cdot \text{CTR}.$$

**Política de los logs.** Random, pero esta vez uniforme de verdad, es decir, las 35.904 vitrinas ordenadas posibles tienen la misma probabilidad. Así se conoce la probabilidad de cada vitrina y se puede usar el IPW de vitrina.

**Políticas evaluadas.** Eligen la vitrina posición por posición, entre los ítems que quedan, con probabilidad proporcional a
$$\exp\big(\eta \log \theta_1(x, a) + \kappa \cdot \#\{\text{ítems ya elegidos de la misma categoría que } a\}\big).$$
- *populares*, con $\eta = 3$ y $\kappa = 0$. Muestra más los ítems que le gustan al usuario.
- *parecidos*, con $\eta = 3$ y $\kappa = 1{,}5$. Además junta ítems de la misma categoría.

Al final se guarda todo en Contenido/Datasets/dataset_sintetico.npz.

In [2]:
from pathlib import Path
from itertools import permutations
import numpy as np
import pandas as pd

OBD_PATH = Path("Contenido/Datasets/open_bandit_dataset/open_bandit_dataset")
CAMPAIGN = "men"

# CTR de cada ítem y efecto de cada posición, sacados de random en OBD
log = pd.read_csv(OBD_PATH / "random" / CAMPAIGN / f"{CAMPAIGN}.csv", usecols=["item_id", "position", "click"])
item_context = pd.read_csv(OBD_PATH / "random" / CAMPAIGN / "item_context.csv", index_col=0).sort_values("item_id")
categoria = item_context["item_feature_1"].astype("category").cat.codes.to_numpy()
n_items = len(categoria)

ctr = log["click"].mean()
ctr_item = log.groupby("item_id")["click"].mean().reindex(range(n_items)).to_numpy()
efecto_pos = log.groupby("position")["click"].mean().sort_index().to_numpy() / ctr

pd.DataFrame({"CTR": [ctr], "CTR mínimo de un ítem": [ctr_item.min()], "CTR máximo de un ítem": [ctr_item.max()],
              "Efecto posición 1": [efecto_pos[0]], "Efecto posición 2": [efecto_pos[1]],
              "Efecto posición 3": [efecto_pos[2]]}).round(5)

,CTR,CTR mínimo de un ítem,CTR máximo de un ítem,Efecto posición 1,Efecto posición 2,Efecto posición 3
0,0.00512,0.00316,0.00765,1.04987,1.02353,0.92662


## Construcción

Se calcula $\theta$ para cada tipo de usuario, ítem y posición, se arman las 35.904 vitrinas posibles con sus vecinos de la misma categoría, y para cada política evaluada se calcula la probabilidad de cada vitrina para cada tipo de usuario. Con eso también se obtiene $\pi_e(a \mid x, k)$ y cuántos vecinos de su categoría tendría cada ítem en cada posición con $\pi_e$.

In [3]:
def contar_vecinos(cat):
    # cuántos de los otros dos ítems de la vitrina son de la misma categoría
    igual_12 = (cat[:, 0] == cat[:, 1]).astype(int)
    igual_13 = (cat[:, 0] == cat[:, 2]).astype(int)
    igual_23 = (cat[:, 1] == cat[:, 2]).astype(int)
    return np.column_stack([igual_12 + igual_13, igual_12 + igual_23, igual_13 + igual_23])


N_TIPOS, DIM, SIGMA = 20, 5, 0.25
rng = np.random.default_rng(0)
x_tipo = rng.normal(size=(N_TIPOS, DIM))           # características de cada tipo de usuario
v_item = rng.normal(size=(n_items, DIM))
afinidad = x_tipo @ v_item.T / np.sqrt(DIM)
theta_sint = (ctr_item[None, :, None] * efecto_pos[None, None, :]
              * np.exp(SIGMA * afinidad - SIGMA ** 2 / 2)[:, :, None])     # (tipo, ítem, posición)

# todas las vitrinas ordenadas posibles y sus vecinos
todas = np.array(list(permutations(range(n_items), 3)))
vecinos_todas = contar_vecinos(categoria[todas])
K = np.arange(3)


def politica_evaluada(eta, kappa):
    # probabilidad de cada vitrina para cada tipo de usuario, eligiendo posición por posición
    prob = np.empty((N_TIPOS, len(todas)))
    filas = np.arange(len(todas))
    for u in range(N_TIPOS):
        puntaje = eta * np.log(theta_sint[u, :, 0])
        log_p = np.zeros(len(todas))
        usado = np.zeros((len(todas), n_items), dtype=bool)
        misma_cat = np.zeros((len(todas), n_items))
        for k in range(3):
            l = np.where(usado, -np.inf, puntaje[None, :] + kappa * misma_cat)
            maximo = l.max(axis=1)
            log_p += l[filas, todas[:, k]] - maximo - np.log(np.exp(l - maximo[:, None]).sum(axis=1))
            usado[filas, todas[:, k]] = True
            misma_cat += categoria[None, :] == categoria[todas[:, k]][:, None]
        prob[u] = np.exp(log_p)
    return prob


def resumen_politica(prob):
    # pi_e(a | x, k) y vecinos esperados de cada ítem en cada posición con pi_e
    pi = np.zeros((N_TIPOS, n_items, 3))
    vec = np.zeros((N_TIPOS, n_items, 3))
    for u in range(N_TIPOS):
        for k in range(3):
            pi[u, :, k] = np.bincount(todas[:, k], prob[u], n_items)
            vec[u, :, k] = np.bincount(todas[:, k], prob[u] * vecinos_todas[:, k], n_items) / pi[u, :, k]
    return pi, vec


politicas_sint = {}
for nombre, (eta, kappa) in {"populares": (3, 0), "parecidos": (3, 1.5)}.items():
    prob = politica_evaluada(eta, kappa)
    politicas_sint[nombre] = (prob, *resumen_politica(prob))

pd.DataFrame({
    "random uniforme (logs)": {"Vecinos de la misma categoría por vitrina": vecinos_todas.sum(axis=1).mean()},
    **{nombre: {"Vecinos de la misma categoría por vitrina": (prob @ vecinos_todas.sum(axis=1)).mean()}
       for nombre, (prob, _, _) in politicas_sint.items()},
}).round(3)

,random uniforme (logs),populares,parecidos
Vecinos de la misma categoría por vitrina,0.834,0.982,2.625


El $\theta$ más bajo es cerca de 0,33 veces el CTR. Por eso, con el modelo lineal, $s$ no puede pasar de 0,15, porque con dos vecinos la probabilidad de clic del ítem menos clickeado quedaría negativa.

In [4]:
pd.Series({"theta mínimo / CTR": theta_sint.min() / ctr, "theta máximo / CTR": theta_sint.max() / ctr,
           "s máximo con el modelo lineal": theta_sint.min() / ctr / 2}).round(3)

theta mínimo / CTR               0.334
theta máximo / CTR               3.184
s máximo con el modelo lineal    0.167
dtype: float64

## Guardar el dataset

Se guarda todo lo que hace falta para generar logs, es decir, $\theta$, las características de los usuarios, las categorías, las vitrinas posibles con sus vecinos, el CTR de referencia (para $\beta = s \cdot \text{CTR}$) y, para cada política evaluada, la probabilidad de cada vitrina, $\pi_e(a \mid x, k)$ y los vecinos esperados.

In [5]:
DATA_PATH = Path("Contenido/Datasets")
guardar = {"theta": theta_sint, "x_tipo": x_tipo, "categoria": categoria, "todas": todas,
           "vecinos_todas": vecinos_todas, "ctr": np.array(ctr), "politicas": np.array(list(politicas_sint))}
for nombre, (prob, pi_e, vec_e) in politicas_sint.items():
    guardar[f"prob_{nombre}"], guardar[f"pi_e_{nombre}"], guardar[f"vec_e_{nombre}"] = prob, pi_e, vec_e
np.savez_compressed(DATA_PATH / "dataset_sintetico.npz", **guardar)
print(f"Guardado en {DATA_PATH / 'dataset_sintetico.npz'}")

Guardado en Contenido/Datasets/dataset_sintetico.npz


## Generar logs

Un log de $n$ vitrinas se genera sorteando un tipo de usuario y una vitrina uniforme para cada fila, y los clics con $\theta - \beta c$. La función log_formato_obp entrega el log con el mismo formato que obtain_batch_bandit_feedback de obp, con una fila por impresión, con context, action, position (de 0 a 2), reward y pscore. También entrega action_dist de la política evaluada, con forma (filas, ítems, posiciones), y su CTR real, que se calcula de forma exacta.

In [6]:
K = np.arange(3)
P_LOGS_POS = 1 / n_items          # probabilidad de cada ítem en cada posición en los logs


def simular_log_sint(s, n, semilla):
    rng = np.random.default_rng(semilla)
    u = rng.integers(N_TIPOS, size=n)                       # tipo de usuario de cada fila
    idx = rng.integers(len(todas), size=n)                  # vitrina uniforme
    vit, vec = todas[idx], vecinos_todas[idx]
    r = (rng.random((n, 3)) < theta_sint[u[:, None], vit, K] - s * ctr * vec).astype(float)
    return u, idx, vit, vec, r


def ctr_real(nombre, s):
    # CTR por impresión de la política evaluada, exacto
    prob = politicas_sint[nombre][0]
    return np.mean([(prob[u][:, None] * (theta_sint[u][todas, K] - s * ctr * vecinos_todas)).sum() / 3
                    for u in range(N_TIPOS)])


def log_formato_obp(nombre, s, n, semilla):
    u, idx, vit, vec, r = simular_log_sint(s, n, semilla)
    pi_e = politicas_sint[nombre][1]
    u_fila = np.repeat(u, 3)
    feedback = {
        "n_rounds": 3 * n,
        "n_actions": n_items,
        "context": x_tipo[u_fila],
        "action_context": np.eye(n_items),
        "action": vit.ravel(),
        "position": np.tile(K, n),
        "reward": r.ravel().astype(int),
        "pscore": np.full(3 * n, P_LOGS_POS),
        "vitrina": np.repeat(np.arange(n), 3),               # para saber qué filas van juntas
        "vecinos": vec.ravel(),
    }
    action_dist = pi_e[u_fila]                               # (filas, ítems, posiciones)
    return feedback, action_dist, ctr_real(nombre, s)


feedback, action_dist, ctr_real_sint = log_formato_obp("parecidos", 0.10, 3_334, 0)
# revisión, el IPW de obp es el promedio de reward * action_dist[fila, action, position] / pscore
ipw_obp = (feedback["reward"] * action_dist[np.arange(feedback["n_rounds"]), feedback["action"], feedback["position"]]
           / feedback["pscore"]).mean()
pd.Series({"Filas": feedback["n_rounds"], "CTR del log": feedback["reward"].mean(),
           "IPW (como obp)": ipw_obp, "CTR real de la política": ctr_real_sint}).round(5)

Filas                      10002.00000
CTR del log                    0.00510
IPW (como obp)                 0.00583
CTR real de la política        0.00677
dtype: float64

**Declaración de uso de IA**

Utilicé IA (Claude versión gratis) para que se entienda de forma correcta lo que hace cada parte del código y la construcción teórica matemática de cada segmento (markdowns) y para tratar inicialmente los datos (recuperar la información relevante al momento de cargar los datos). Además, se utilizó como apoyo en el código, tanto para crear el código del dataset sintético como para arreglar errores y hacer ajustes.